# aiprom-llm · Reproducible MLX Workflow for FHIR Questionnaire Generation

This notebook documents the repository's current end-to-end workflow for preparing data, materializing deterministic training artifacts, previewing MLX-LM training commands, and optionally generating downstream artifacts for complete FHIR R4 Questionnaire generation.

It is written as a publication-oriented technical document: the checked-in outputs correspond to the current executed 7B 8-bit release snapshot, while all rerun-sensitive helper logic is centralized in reusable Python modules under `scripts/`.

## Current supported scope

- Dataset: `data/synthetic-aiprom-1500-firh4.jsonl`
- Supported configs: `configs/Qwen2.5-Coder-7B-Instruct-bf16.yaml`, `configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml`, and `configs/Qwen2.5-Coder-3B-Instruct-bf16.yaml`
- Current best checked-in snapshot: `mlx-community/Qwen2.5-Coder-7B-Instruct-8bit` with published `GO` evaluation
- Training backend: `mlx_lm` on Apple Silicon
- Shared artifacts: `lab/artifacts/`
- Model-scoped artifacts: `lab/artifacts/<MODEL_ALIAS>/`

This notebook presents the supported workflow in the repository as it exists today. It should be read as a reproducible engineering workflow and as the archived executed snapshot behind the current checked-in release evidence.

## How To Use This Notebook

This document is intended to be readable from top to bottom without requiring a prior codebase deep dive.

### Execution policy

- Run the configuration cell first.
- Training, export, inference, and evaluation are disabled by default.
- No section should overwrite artifacts unless you explicitly opt in by changing a control flag.
- The notebook uses only repository-relative paths.

### Publication policy

- The notebook documents only the workflow supported by the current repository snapshot.
- Optional artifacts such as A/B evaluation reports or go/no-go summaries are treated as conditional outputs, not guaranteed deliverables.
- Transient runtime logs are not treated as publication artifacts.

### Environment assumptions

- Launch Jupyter from the repository root.
- Use the Poetry environment defined in `pyproject.toml`.
- Keep checked-in YAML configs as the single source of truth for model and training settings.

## Section Map

The notebook follows this publication-oriented order:

1. Context and execution contract
2. Dataset validation and governance
3. Deterministic split and materialization
4. Configuration and artifact plan
5. Training command preview and optional execution
6. GGUF export plan and external tooling preconditions
7. Inference preview and optional execution
8. Complete A/B evaluation and artifact generation
9. Limitations and publication status

This ordering keeps the narrative linear while making the expensive steps explicit and opt-in.

## Central Configuration

The next code cell is the single execution control point for this notebook.

It resolves the repository root, selects the active YAML config, derives the model alias, and exposes safe execution flags for materialization, training, export, inference, and evaluation.

All expensive operations default to `False`. Change them deliberately only when you want to execute the corresponding step.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

bootstrap_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "scripts" / "notebook_workflow.py").exists():
        bootstrap_root = candidate
        break

if bootstrap_root is None:
    raise FileNotFoundError("Could not locate scripts/notebook_workflow.py from the current working directory.")

if str(bootstrap_root) not in sys.path:
    sys.path.insert(0, str(bootstrap_root))

from scripts.notebook_workflow import (
    DEFAULT_CONFIG_REL_PATH,
    DEFAULT_DATASET_REL_PATH,
    MaterializedDataset,
    adapter_checkpoint_dir,
    artifact_status,
    build_fuse_command,
    build_generate_command,
    build_llama_cpp_convert_command,
    build_publication_notes,
    build_training_command,
    build_training_snapshot,
    build_workflow_config,
    command_preview,
    extract_json_object,
    load_prompt_completion_records,
    materialize_dataset,
    persist_training_snapshot,
    read_json_if_exists,
    run_ab_evaluation,
    run_fhir_audit,
    sample_validation_records,
    stratified_train_val_test_split,
    summarize_dataset,
    summarize_evaluation_payloads,
    summarize_split_distribution,
    workflow_summary,
    write_json_artifact,
)

CONFIG_OVERRIDES = {
    "AIPROM_CONFIG": "configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml",
    "AIPROM_DATASET": DEFAULT_DATASET_REL_PATH,
    "AIPROM_RUN_MATERIALIZATION": "0",
    "AIPROM_RUN_TRAINING": "0",
    "AIPROM_RUN_EXPORT": "0",
    "AIPROM_RUN_ADAPTER_INFERENCE": "0",
    "AIPROM_RUN_BASELINE_INFERENCE": "0",
    "AIPROM_RUN_EVALUATION": "0",
    "AIPROM_UPDATE_TUNES_REPORTS": "0",
    "AIPROM_EVAL_SAMPLES": "30",
}

WORKFLOW = build_workflow_config(CONFIG_OVERRIDES)
EVAL_SAMPLES = int(CONFIG_OVERRIDES.get("AIPROM_EVAL_SAMPLES", "30"))
pd.set_option("display.max_colwidth", 120)

summary_rows = [
    {"field": key, "value": value}
    for key, value in workflow_summary(WORKFLOW).items()
    if key not in {"repo_root", "shared_artifacts_dir", "model_artifacts_dir"}
]

display(pd.DataFrame(summary_rows))
print(f"repo_root: {WORKFLOW.repo_root}")
print(f"shared_artifacts_dir: {WORKFLOW.shared_artifacts_dir}")
print(f"model_artifacts_dir: {WORKFLOW.model_artifacts_dir}")
print(f"eval_samples: {EVAL_SAMPLES}")

,field,value
0,dataset_path,/Users/CAE9/aiprom-llm/data/synthetic-aiprom-1500-firh4.jsonl
1,config_path,/Users/CAE9/aiprom-llm/configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml
2,model_name,mlx-community/Qwen2.5-Coder-7B-Instruct-8bit
3,model_alias,mlx-community-qwen2.5-coder-7b-instruct-8bit
4,model_backend,mlx_lm
5,global_seed,173
6,run_materialization,False
7,run_training,False
8,run_export,False
9,run_adapter_inference,False


repo_root: /Users/CAE9/aiprom-llm
shared_artifacts_dir: /Users/CAE9/aiprom-llm/lab/artifacts
model_artifacts_dir: /Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit
eval_samples: 30


In [2]:
publication_notes = pd.DataFrame(build_publication_notes(WORKFLOW))
artifact_inventory = pd.DataFrame(
    [
        {"artifact": name, **details}
        for name, details in artifact_status(WORKFLOW).items()
    ]
).sort_values("artifact").reset_index(drop=True)

display(publication_notes)
display(artifact_inventory)

,topic,note
0,Dataset,Primary dataset: data/synthetic-aiprom-1500-firh4.jsonl
1,Config,Active YAML config: configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml
2,Artifacts,Shared artifacts: lab/artifacts
3,Model artifacts,Model-scoped artifacts: lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit
4,Execution policy,"Training, export, inference and evaluation are disabled by default."


,artifact,path,exists
0,ab_rule_eval,/Users/CAE9/aiprom-llm/lab/artifacts/ab_rule_eval.json,True
1,ab_rule_eval_analysis,/Users/CAE9/aiprom-llm/lab/artifacts/ab_rule_eval_analysis.json,True
2,adapter_checkpoint_dir,/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit/checkpoints_stable,True
3,adapter_go_no_go,/Users/CAE9/aiprom-llm/lab/artifacts/adapter_go_no_go.json,True
4,dataset_manifest,/Users/CAE9/aiprom-llm/lab/artifacts/dataset_manifest.json,True
5,fused_noquant_dir,/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit/fused-noquant,True
6,split_manifest,/Users/CAE9/aiprom-llm/lab/artifacts/split_manifest.json,True
7,training_config,/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit/training_config_stable.json,True
8,tune_report,/Users/CAE9/aiprom-llm/tunes/models/mlx-community-qwen2.5-coder-7b-instruct-8bit.md,True


## Dataset Statement And Structural Validation

This repository currently trains on a single prompt/completion dataset: `data/synthetic-aiprom-1500-firh4.jsonl`. Each row contains a natural-language prompt and a JSON-encoded FHIR R4 `Questionnaire` completion.

### What this section checks

- dataset path resolution from the repository root
- JSONL and completion parsing
- minimal Questionnaire structural validity
- item-type coverage in the current committed snapshot

### Governance note

This notebook is for research and engineering reproducibility. Some questionnaire content may reflect third-party instruments with their own licensing terms. For redistribution or commercial use, review `README.md` and `datasets.md` instrument by instrument.

In [3]:
RECORDS, PARSE_ERRORS = load_prompt_completion_records(WORKFLOW.dataset_path)
if PARSE_ERRORS:
    raise RuntimeError("Dataset parsing failed. Sample errors:\n- " + "\n- ".join(PARSE_ERRORS[:10]))

DATASET_SUMMARY = summarize_dataset(RECORDS)
FHIR_AUDIT = run_fhir_audit(RECORDS)

overview_df = pd.DataFrame(
    [
        {"metric": "dataset_path", "value": str(WORKFLOW.dataset_path)},
        {"metric": "records", "value": DATASET_SUMMARY["records"]},
        {"metric": "min_items_per_form", "value": DATASET_SUMMARY["min_items_per_form"]},
        {"metric": "max_items_per_form", "value": DATASET_SUMMARY["max_items_per_form"]},
        {"metric": "avg_items_per_form", "value": DATASET_SUMMARY["avg_items_per_form"]},
        {"metric": "fhir_pass_rate_percent", "value": FHIR_AUDIT.pass_rate},
        {"metric": "invalid_records", "value": FHIR_AUDIT.invalid_records},
    ]
)

item_type_df = pd.DataFrame(
    [
        {"item_type": item_type, "count": count}
        for item_type, count in DATASET_SUMMARY["item_type_counts"].items()
    ]
).sort_values("count", ascending=False).reset_index(drop=True)

issue_df = pd.DataFrame(
    [
        {"issue": issue, "count": count}
        for issue, count in FHIR_AUDIT.issue_counts.items()
    ]
)

display(overview_df)
display(item_type_df)
display(issue_df if not issue_df.empty else pd.DataFrame([{"issue": "none", "count": 0}]))

,metric,value
0,dataset_path,/Users/CAE9/aiprom-llm/data/synthetic-aiprom-1500-firh4.jsonl
1,records,1500
2,min_items_per_form,5
3,max_items_per_form,9
4,avg_items_per_form,5.8
5,fhir_pass_rate_percent,100.0
6,invalid_records,0


,item_type,count
0,integer,3450
1,choice,3300
2,boolean,1350
3,decimal,450
4,string,150


,issue,count
0,none,0


## Deterministic Split And Materialization

The repository uses a deterministic stratified split based on the dominant item type per questionnaire. This keeps the split logic explicit while matching the semantics documented in `datasets.md`.

### Publication-safe behavior

- The split is always computed in memory for inspection.
- JSONL materialization is optional and disabled by default.
- When enabled, the notebook writes only the standard shared artifacts under `lab/artifacts/`: `split_manifest.json`, `train.jsonl`, `val.jsonl`, `valid.jsonl`, and `dataset_manifest.json`.

## Artifact Naming And Model Alias

The workflow uses a single canonical alias for model-scoped artifacts: a slugified form of the model identifier, for example `mlx-community-qwen2.5-coder-7b-instruct-bf16`.

This same alias now aligns across:

- the notebook workflow
- `lab/artifacts/<MODEL_ALIAS>/...`
- the checked-in YAML configs in `configs/`
- tune reports under `tunes/models/`

That alignment avoids the older ambiguity between mixed-case adapter paths in YAML and slugified artifact directories in the notebook.

In [4]:
VAL_RATIO = 0.15
TEST_RATIO = 0.15

TRAIN_INDICES, VAL_INDICES, TEST_INDICES = stratified_train_val_test_split(
    RECORDS,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=WORKFLOW.global_seed,
)

split_distribution_df = pd.DataFrame(
    [
        {"split": "train", "size": len(TRAIN_INDICES), "distribution": summarize_split_distribution(RECORDS, TRAIN_INDICES)},
        {"split": "validation", "size": len(VAL_INDICES), "distribution": summarize_split_distribution(RECORDS, VAL_INDICES)},
        {"split": "test", "size": len(TEST_INDICES), "distribution": summarize_split_distribution(RECORDS, TEST_INDICES)},
    ]
)

materialization_plan = MaterializedDataset(
    train_path=WORKFLOW.shared_artifacts_dir / "train.jsonl",
    val_path=WORKFLOW.shared_artifacts_dir / "val.jsonl",
    valid_path=WORKFLOW.shared_artifacts_dir / "valid.jsonl",
    dataset_manifest_path=WORKFLOW.shared_artifacts_dir / "dataset_manifest.json",
    split_manifest_path=WORKFLOW.shared_artifacts_dir / "split_manifest.json",
    train_size=len(TRAIN_INDICES),
    val_size=len(VAL_INDICES),
    test_size=len(TEST_INDICES),
)

if WORKFLOW.run_materialization:
    MATERIALIZED = materialize_dataset(
        WORKFLOW,
        RECORDS,
        train_indices=TRAIN_INDICES,
        val_indices=VAL_INDICES,
        test_indices=TEST_INDICES,
        seed=WORKFLOW.global_seed,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
    )
    materialization_status = "materialized"
else:
    MATERIALIZED = materialization_plan
    materialization_status = "preview_only"

plan_df = pd.DataFrame(
    [
        {"artifact": "split_manifest", "path": str(MATERIALIZED.split_manifest_path), "exists_now": MATERIALIZED.split_manifest_path.exists()},
        {"artifact": "train_jsonl", "path": str(MATERIALIZED.train_path), "exists_now": MATERIALIZED.train_path.exists()},
        {"artifact": "val_jsonl", "path": str(MATERIALIZED.val_path), "exists_now": MATERIALIZED.val_path.exists()},
        {"artifact": "valid_jsonl", "path": str(MATERIALIZED.valid_path), "exists_now": MATERIALIZED.valid_path.exists()},
        {"artifact": "dataset_manifest", "path": str(MATERIALIZED.dataset_manifest_path), "exists_now": MATERIALIZED.dataset_manifest_path.exists()},
    ]
)

print(f"materialization_status: {materialization_status}")
display(split_distribution_df)
display(plan_df)

materialization_status: preview_only


,split,size,distribution
0,train,1048,"{'boolean': 314, 'choice': 314, 'integer': 420}"
1,validation,226,"{'boolean': 68, 'choice': 68, 'integer': 90}"
2,test,226,"{'boolean': 68, 'choice': 68, 'integer': 90}"


,artifact,path,exists_now
0,split_manifest,/Users/CAE9/aiprom-llm/lab/artifacts/split_manifest.json,True
1,train_jsonl,/Users/CAE9/aiprom-llm/lab/artifacts/train.jsonl,True
2,val_jsonl,/Users/CAE9/aiprom-llm/lab/artifacts/val.jsonl,True
3,valid_jsonl,/Users/CAE9/aiprom-llm/lab/artifacts/valid.jsonl,True
4,dataset_manifest,/Users/CAE9/aiprom-llm/lab/artifacts/dataset_manifest.json,True


In [5]:
TRAINING_SNAPSHOT = build_training_snapshot(WORKFLOW, MATERIALIZED)
TRAINING_SNAPSHOT_PATH = WORKFLOW.model_artifacts_dir / "training_config_stable.json"
TRAINING_COMMAND = build_training_command(WORKFLOW)

yaml_cfg = TRAINING_SNAPSHOT["config_yaml"]
training_overview_df = pd.DataFrame(
    [
        {"field": "config_path", "value": str(WORKFLOW.config_path)},
        {"field": "model", "value": WORKFLOW.model_name},
        {"field": "model_alias", "value": WORKFLOW.model_alias},
        {"field": "seed", "value": WORKFLOW.global_seed},
        {"field": "iters", "value": yaml_cfg.get("iters")},
        {"field": "batch_size", "value": yaml_cfg.get("batch_size")},
        {"field": "grad_accumulation_steps", "value": yaml_cfg.get("grad_accumulation_steps")},
        {"field": "max_seq_length", "value": yaml_cfg.get("max_seq_length")},
        {"field": "learning_rate", "value": yaml_cfg.get("learning_rate")},
        {"field": "checkpoint_dir", "value": TRAINING_SNAPSHOT["outputs"]["checkpoint_dir"]},
        {"field": "snapshot_path", "value": str(TRAINING_SNAPSHOT_PATH)},
    ]
)

display(training_overview_df)
print(command_preview(TRAINING_COMMAND))

,field,value
0,config_path,/Users/CAE9/aiprom-llm/configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml
1,model,mlx-community/Qwen2.5-Coder-7B-Instruct-8bit
2,model_alias,mlx-community-qwen2.5-coder-7b-instruct-8bit
3,seed,173
4,iters,1000
5,batch_size,1
6,grad_accumulation_steps,8
7,max_seq_length,1024
8,learning_rate,5e-6
9,checkpoint_dir,/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit/checkpoints_stable


poetry run python -m mlx_lm lora -c configs/Qwen2.5-Coder-7B-Instruct-8bit.yaml


In [6]:
training_result = None

if not WORKFLOW.run_training:
    print("Training execution is disabled. Set AIPROM_RUN_TRAINING=1 in CONFIG_OVERRIDES to persist the snapshot and run MLX-LM.")
else:
    persisted_path = persist_training_snapshot(TRAINING_SNAPSHOT)
    training_process = subprocess.run(
        TRAINING_COMMAND,
        cwd=str(WORKFLOW.repo_root),
        text=True,
        capture_output=True,
        check=False,
    )
    training_result = {
        "return_code": training_process.returncode,
        "stdout_tail": training_process.stdout[-1200:],
        "stderr_tail": training_process.stderr[-1200:],
        "training_config_path": str(persisted_path),
    }
    if training_process.returncode != 0:
        raise RuntimeError(training_result["stderr_tail"] or "Training command failed.")
    print(pd.DataFrame([training_result]))

Training execution is disabled. Set AIPROM_RUN_TRAINING=1 in CONFIG_OVERRIDES to persist the snapshot and run MLX-LM.


## GGUF Export And External Tooling

The repository documents GGUF export as an optional downstream step, not as part of the minimal published snapshot.

### Preconditions

- A trained adapter must exist under `lab/artifacts/<MODEL_ALIAS>/checkpoints_stable/`.
- `tools/llama.cpp` must be present and built if you want to convert the fused model to GGUF inside this workspace.
- Export remains disabled by default because it is costly and environment-dependent.

The next cell previews the supported commands. It only executes them if you explicitly enable export in the central configuration cell.

In [7]:
FUSE_COMMAND = build_fuse_command(WORKFLOW)
CONVERT_COMMAND = build_llama_cpp_convert_command(WORKFLOW)
llama_cpp_script = WORKFLOW.repo_root / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"

export_plan_df = pd.DataFrame(
    [
        {"step": "adapter_checkpoint_dir", "path": str(adapter_checkpoint_dir(WORKFLOW)), "ready": adapter_checkpoint_dir(WORKFLOW).exists()},
        {"step": "fuse_command", "path": command_preview(FUSE_COMMAND), "ready": True},
        {"step": "llama_cpp_convert_command", "path": command_preview(CONVERT_COMMAND), "ready": llama_cpp_script.exists()},
    ]
)

display(export_plan_df)

if WORKFLOW.run_export:
    if not adapter_checkpoint_dir(WORKFLOW).exists():
        raise FileNotFoundError("Adapter checkpoints were not found. Run training first.")
    fuse_process = subprocess.run(FUSE_COMMAND, cwd=str(WORKFLOW.repo_root), text=True, capture_output=True, check=False)
    if fuse_process.returncode != 0:
        raise RuntimeError(fuse_process.stderr[-1200:] or "Fuse command failed.")
    if llama_cpp_script.exists():
        convert_process = subprocess.run(CONVERT_COMMAND, cwd=str(WORKFLOW.repo_root), text=True, capture_output=True, check=False)
        if convert_process.returncode != 0:
            raise RuntimeError(convert_process.stderr[-1200:] or "GGUF conversion failed.")
        print("Fuse and GGUF conversion completed.")
    else:
        print("Fuse completed. llama.cpp conversion was skipped because tools/llama.cpp is not available.")
else:
    print("Export execution is disabled. The table above is the publication-safe preview.")

,step,path,ready
0,adapter_checkpoint_dir,/Users/CAE9/aiprom-llm/lab/artifacts/mlx-community-qwen2.5-coder-7b-instruct-8bit/checkpoints_stable,True
1,fuse_command,poetry run python -m mlx_lm fuse --model mlx-community/Qwen2.5-Coder-7B-Instruct-8bit --adapter-path /Users/CAE9/aip...,True
2,llama_cpp_convert_command,poetry run python /Users/CAE9/aiprom-llm/tools/llama.cpp/convert_hf_to_gguf.py /Users/CAE9/aiprom-llm/lab/artifacts/...,True


Export execution is disabled. The table above is the publication-safe preview.


In [8]:
INFERENCE_PROMPT = (
    "Generate a complete FHIR Questionnaire for a weekly recovery check-in form. "
    "Include resourceType='Questionnaire', status='active', a title, and six items with mixed types."
)

BASELINE_COMMAND = build_generate_command(
    WORKFLOW,
    user_prompt=INFERENCE_PROMPT,
    use_adapter=False,
)
ADAPTER_COMMAND = build_generate_command(
    WORKFLOW,
    user_prompt=INFERENCE_PROMPT,
    use_adapter=True,
)

inference_plan_df = pd.DataFrame(
    [
        {"mode": "baseline", "enabled": WORKFLOW.run_baseline_inference, "command": command_preview(BASELINE_COMMAND)},
        {"mode": "adapter", "enabled": WORKFLOW.run_adapter_inference, "command": command_preview(ADAPTER_COMMAND)},
    ]
)
display(inference_plan_df)

INFERENCE_RESULTS = []
for mode, enabled, command in [
    ("baseline", WORKFLOW.run_baseline_inference, BASELINE_COMMAND),
    ("adapter", WORKFLOW.run_adapter_inference, ADAPTER_COMMAND),
]:
    if not enabled:
        continue
    result = subprocess.run(command, cwd=str(WORKFLOW.repo_root), text=True, capture_output=True, check=False)
    generated_json, parse_error = extract_json_object(result.stdout)
    INFERENCE_RESULTS.append(
        {
            "mode": mode,
            "return_code": result.returncode,
            "json_extracted": generated_json is not None,
            "parse_error": parse_error,
            "stdout_preview": result.stdout[:300],
        }
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-1200:] or f"{mode} inference failed.")

if INFERENCE_RESULTS:
    display(pd.DataFrame(INFERENCE_RESULTS))
else:
    print("Inference execution is disabled. The command previews above are the default publication state.")

,mode,enabled,command
0,baseline,False,poetry run python -m mlx_lm generate --ignore-chat-template --model mlx-community/Qwen2.5-Coder-7B-Instruct-8bit --p...
1,adapter,False,poetry run python -m mlx_lm generate --ignore-chat-template --model mlx-community/Qwen2.5-Coder-7B-Instruct-8bit --p...


Inference execution is disabled. The command previews above are the default publication state.


## Optional A/B Evaluation And Artifact Generation

This section can generate the repository's complete A/B evaluation artifact set when enabled.

### What it does

- samples validation prompts deterministically from the split defined earlier
- runs the same prompts through the base model and the adapter
- scores both sides with the same structural rule set
- writes `ab_rule_eval.json`, `ab_rule_eval_analysis.json`, and `adapter_go_no_go.json` under `lab/artifacts/`

### Execution policy

- Evaluation stays disabled by default.
- When disabled, the next cell only previews or summarizes existing artifacts.
- When enabled, it performs the full baseline-versus-adapter loop and produces the complete A/B artifact set needed by the reporting pipeline.

In [9]:
validation_records = sample_validation_records(
    RECORDS,
    VAL_INDICES,
    limit=EVAL_SAMPLES,
    seed=WORKFLOW.global_seed,
 )

evaluation_preview_df = pd.DataFrame(
    [
        {
            "mode": "baseline",
            "enabled": WORKFLOW.run_evaluation,
            "command_template": command_preview(
                build_generate_command(
                    WORKFLOW,
                    user_prompt=validation_records[0].prompt if validation_records else "Generate a complete FHIR Questionnaire.",
                    use_adapter=False,
                )
            ),
        },
        {
            "mode": "adapter",
            "enabled": WORKFLOW.run_evaluation,
            "command_template": command_preview(
                build_generate_command(
                    WORKFLOW,
                    user_prompt=validation_records[0].prompt if validation_records else "Generate a complete FHIR Questionnaire.",
                    use_adapter=True,
                )
            ),
        },
    ]
)
display(evaluation_preview_df)

rule_eval_path = WORKFLOW.shared_artifacts_dir / "ab_rule_eval.json"
analysis_path = WORKFLOW.shared_artifacts_dir / "ab_rule_eval_analysis.json"
decision_path = WORKFLOW.shared_artifacts_dir / "adapter_go_no_go.json"

if WORKFLOW.run_evaluation:
    rule_eval_payload, analysis_payload, decision_payload = run_ab_evaluation(
        WORKFLOW,
        validation_records=validation_records,
        materialized=MATERIALIZED,
    )
    write_json_artifact(rule_eval_path, rule_eval_payload)
    write_json_artifact(analysis_path, analysis_payload)
    write_json_artifact(decision_path, decision_payload)

    if WORKFLOW.update_tunes_reports:
        report_process = subprocess.run(
            [
                "poetry",
                "run",
                "python",
                "scripts/update_tunes_reports.py",
                "--repo-root",
                ".",
                "--model-name",
                WORKFLOW.model_name,
            ],
            cwd=str(WORKFLOW.repo_root),
            text=True,
            capture_output=True,
            check=False,
        )
        if report_process.returncode != 0:
            raise RuntimeError(report_process.stderr[-1200:] or "Failed to update tune reports.")
        print("Updated tune reports and leaderboard from the new evaluation artifacts.")

    display(pd.DataFrame(summarize_evaluation_payloads(rule_eval_payload, analysis_payload, decision_payload)))
    print(f"Saved: {rule_eval_path}")
    print(f"Saved: {analysis_path}")
    print(f"Saved: {decision_path}")
else:
    rule_eval_payload = read_json_if_exists(rule_eval_path)
    analysis_payload = read_json_if_exists(analysis_path)
    decision_payload = read_json_if_exists(decision_path)
    evaluation_status_df = pd.DataFrame(
        [
            {"artifact": "ab_rule_eval", "available": rule_eval_payload is not None},
            {"artifact": "ab_rule_eval_analysis", "available": analysis_payload is not None},
            {"artifact": "adapter_go_no_go", "available": decision_payload is not None},
        ]
    )
    display(evaluation_status_df)
    if rule_eval_payload or analysis_payload or decision_payload:
        display(pd.DataFrame(summarize_evaluation_payloads(rule_eval_payload, analysis_payload, decision_payload)))
    else:
        print(
            "No complete A/B artifacts are available yet. "
            "Set AIPROM_RUN_EVALUATION=1 in CONFIG_OVERRIDES to generate ab_rule_eval.json, ab_rule_eval_analysis.json, and adapter_go_no_go.json."
        )

,mode,enabled,command_template
0,baseline,False,poetry run python -m mlx_lm generate --ignore-chat-template --model mlx-community/Qwen2.5-Coder-7B-Instruct-8bit --p...
1,adapter,False,poetry run python -m mlx_lm generate --ignore-chat-template --model mlx-community/Qwen2.5-Coder-7B-Instruct-8bit --p...


,artifact,available
0,ab_rule_eval,True
1,ab_rule_eval_analysis,True
2,adapter_go_no_go,True


,metric,value
0,baseline_samples,30
1,adapter_samples,30
2,strict_baseline_rate,13.33
3,strict_adapter_rate,90.0
4,relaxed_baseline_rate,53.33
5,relaxed_adapter_rate,100.0
6,go_no_go_decision,GO


## Limitations, Publication Status, And References

### What this notebook now guarantees

- A linear, repository-aligned narrative for the supported workflow.
- A single control cell for safe execution flags and config selection.
- Repository-relative paths and deterministic split logic.
- Reusable helper logic moved out of the notebook and into `scripts/notebook_workflow.py`.
- A complete optional A/B evaluation path that can generate `ab_rule_eval.json`, `ab_rule_eval_analysis.json`, and `adapter_go_no_go.json`.
- Checked-in outputs aligned to the current executed `mlx-community/Qwen2.5-Coder-7B-Instruct-8bit` snapshot with published `GO` evaluation evidence.

### What it still does not claim automatically

- It does not claim that rerunning this notebook with the current control flags will retrain or reevaluate the model; those actions are intentionally disabled in the published snapshot.
- It does not assume that `llama.cpp` is vendored or built in every checkout.
- It does not treat GGUF export as part of the validated 7B 8-bit release path.

### Practical publication reading

- Read this notebook as portfolio-ready, reproducibility-oriented, and aligned to the final checked-in release snapshot.
- Treat the current checked-in outputs as archived evidence for the winning 7B 8-bit run, with reruns left opt-in from the control cell.

### References

- HL7 FHIR R4 Questionnaire: https://hl7.org/fhir/R4/questionnaire.html
- MLX: https://github.com/ml-explore/mlx
- MLX-LM: https://github.com/ml-explore/mlx-lm
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- FAIR principles: https://www.go-fair.org/fair-principles/